# Find the reviewed epoch82 checkpoint in Google Drive

Use this after the project-folder search returned `not_found`. This cell uses Google Drive API read operations to search accessible active and trashed checkpoint candidates without a folder restriction. It asks for Google authentication; choose the same account used for training.

Only a file with the exact reviewed SHA-256 is copied to `recovered_checkpoints/chase_shape_epoch082`. The source file and training state are not modified. If trash access is blocked, the report provides file links for manual inspection. Search is limited to documented filename prefixes; it cannot prove that no arbitrarily renamed or permanently deleted copy exists.

A verified recovered checkpoint is passed to the already saved recovery inference launcher. Select a T4 runtime to continue test prediction immediately. No model training is started. Missing files or API access problems produce a report instead of a model-substitution attempt.


In [ ]:
# Find the reviewed epoch82 file in Drive (including trash), then resume inference.
from pathlib import Path
from google.colab import auth, drive, files
import google.auth
from googleapiclient.discovery import build
import hashlib, importlib.util, json, os, re, signal, subprocess, sys, urllib.request

BASE = Path('/content/drive/MyDrive/filament-segmentation-2026')
TRAINING_ROOT = BASE / 'fixed_split_experiments/chase_shape_best84_recipe_t4_v1'
OUTPUT = BASE / 'submissions/chase_shape_epoch082_native_t4_recovery_v2'
RECOVERY = BASE / 'recovered_checkpoints/chase_shape_epoch082'
HELPER_COMMIT = '5f87e09fc51dbe91f599450c3c7cab20d343f723'
HELPER_SHA = '4ca42713de1fa1411603f7a9ec207bfa1fee23dffdf1fb8abf94f2e126846680'

def sha(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def main():
    drive.mount('/content/drive')
    helper_path = Path('/content/solar_chase_drive_recovery.py')
    if not helper_path.is_file() or sha(helper_path) != HELPER_SHA:
        url = f'https://raw.githubusercontent.com/ikjoo5/solar_filament/{HELPER_COMMIT}/tools/chase_shape_drive_recovery.py'
        with urllib.request.urlopen(url, timeout=45) as response:
            payload = response.read()
        if hashlib.sha256(payload).hexdigest() != HELPER_SHA:
            raise ValueError('Recovery source checksum differs; use the published notebook.')
        helper_path.write_bytes(payload)
    spec = importlib.util.spec_from_file_location('drive_checkpoint_recovery', helper_path)
    helper = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(helper)
    try:
        auth.authenticate_user()
        credentials, _ = google.auth.default()
        service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
    except Exception as error:
        print('Google Drive API authentication/setup failed:', type(error).__name__)
        print('Use the training Google account. Manual trash check: https://drive.google.com/drive/u/0/trash')
        return
    report = helper.recover(service, RECOVERY)
    print('Recovery status:', report['status'])
    print('Report:', report['report_path'])
    if report['status'] != 'recovered':
        print(report.get('message', ''))
        for item in report.get('candidates', [])[:20]:
            print('Candidate:', item['name'], 'trash=' + str(item.get('trashed', False)), item['webViewLink'])
        print('No training or model inference was started. Keep the report for review.')
        files.download(report['report_path'])
        return report
    checkpoint = Path(report['checkpoint'])
    if sha(checkpoint) != helper.CHECKPOINT_SHA:
        raise ValueError('Recovered weight file changed.')
    print('Verified epoch82 weights:', checkpoint)
    # Reuse the already saved recovery launcher; do not modify training state.
    pointer = OUTPUT / 'test_bootstrap_state.json'
    if not pointer.is_file():
        print('Weights recovered. Open the test notebook and set CHECKPOINT to:', repr(str(checkpoint)))
        return report
    state = json.loads(pointer.read_text())
    if pointer.is_symlink() or state.get('output_root') != str(OUTPUT.resolve()):
        raise ValueError('Saved inference pointer differs.')
    version = state.get('version', '')
    if not re.fullmatch('[0-9a-f]{16}', version):
        raise ValueError('Invalid saved source version.')
    snapshot = OUTPUT / 'code' / version
    manifest_path = snapshot / 'source_manifest.json'
    if not snapshot.resolve().is_relative_to(OUTPUT.resolve()) or manifest_path.is_symlink() or sha(manifest_path) != state['manifest_sha256']:
        raise ValueError('Saved inference manifest changed.')
    manifest = json.loads(manifest_path.read_text())
    canonical = json.dumps(manifest, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()
    if hashlib.sha256(canonical).hexdigest()[:16] != version:
        raise ValueError('Saved inference version differs.')
    for name, expected in manifest.items():
        path = snapshot / name
        if path.is_symlink() or not path.resolve().is_relative_to(snapshot.resolve()) or sha(path) != expected:
            raise ValueError('Saved inference source changed: ' + name)
    bootstrap = snapshot / 'tools/chase_shape_test_colab.py'
    if 'tools/chase_shape_test_colab.py' not in manifest:
        raise ValueError('Saved launcher missing from manifest.')
    command = [sys.executable, '-u', str(bootstrap), '--training-root', str(TRAINING_ROOT),
               '--test-images', str(BASE / 'MAGFiLO_1.0_Kaggle_2026/test/test_images'),
               '--output', str(OUTPUT), '--checkpoint', str(checkpoint), '--search-root', str(BASE)]
    process = subprocess.Popen(command, start_new_session=True, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
        if status:
            raise subprocess.CalledProcessError(status, command)
    finally:
        if process.poll() is None:
            try: os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError: pass
            try: process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                try: os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError: pass
                process.wait()
        process.stdout.close()
    result = json.loads((OUTPUT / 'latest_result.json').read_text())
    if result.get('status') != 'complete' or result.get('images_completed') != 180 or result.get('checkpoint_epoch') != 82 or result.get('checkpoint_sha256') != helper.CHECKPOINT_SHA:
        raise ValueError('Complete verified epoch82 inference is required before download.')
    if Path(result.get('output_root', '')).resolve() != OUTPUT.resolve():
        raise ValueError('Unexpected inference output directory.')
    downloads = []
    for key in ('csv', 'archive'):
        path = Path(result[key])
        if path.is_symlink() or not path.resolve().is_relative_to(OUTPUT.resolve()) or sha(path) != result[key + '_sha256']:
            raise ValueError('Inference output changed: ' + key)
        downloads.append(path)
    for path in downloads:
        print('Download:', path)
        files.download(str(path))
    return result

result = main()
